# Tier 2 + Tier 3: Priority Classifier + Anomaly Detection

**Prerequisite:** run `2.ipynb` first. Reads `cluster_summary_checkpoint.csv`
and `parking_violations_exploded_with_clusters.csv`.

Combined into one notebook since Tier 3's anomaly signal is the natural
companion to Tier 2's prediction -- together they answer both "is this
slot normally risky" (Tier 2) and "is today unusual for this hotspot"
(Tier 3), which is a more complete operational picture than either alone.

## Tier 2: HIGH/LOW Risk Slot Classifier

Binary classifier (not 4-way) -- see prior diagnostic notes: the original
4-way percentile classification scored 33% accuracy / 0.12 F1 on the
CRITICAL class, because each cluster only has ~18-20 (day, time_bucket)
rows on average, too thin for a fine-grained split. Binary HIGH/LOW
against each cluster's own median is a much better-posed problem at this
sample size.

A regression alternative (predicting violation count/impact score
directly) was also tested and rejected: R²=0.49 looked fine on the
surface, but 65% of that came from `total_incident_count` (a per-cluster
constant) rather than real slot-level signal, and 5-fold CV was unstable
(0.39 +/- 0.081) vs. the classifier's tight band (62.7% +/- 2.8%).

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, f1_score

In [2]:
cluster_summary = pd.read_csv("../../data/processed/cluster_summary_checkpoint.csv")
df_clusters = pd.read_csv("../../data/processed/parking_violations_exploded_with_clusters.csv")

print(f"Loaded {len(cluster_summary)} clusters, {len(df_clusters)} violation-level rows")

Loaded 300 clusters, 323431 violation-level rows


In [3]:
df_clusters['cdt_day_of_week'] = pd.to_datetime(df_clusters['created_datetime']).dt.day_name()

def coarse_bucket(hour):
    if 3 <= hour < 7: return 'early_morning'
    if 7 <= hour < 12: return 'late_morning'
    if 12 <= hour < 16: return 'afternoon'
    if 16 <= hour < 20: return 'evening'
    return 'night'

df_clusters['time_bucket'] = df_clusters['cdt_hour_bucket'].apply(coarse_bucket)

slot = (
    df_clusters.groupby(['cluster_id', 'cdt_day_of_week', 'time_bucket'])
    .agg(violation_count=('id', 'nunique'))
    .reset_index()
)
slot = slot.merge(
    cluster_summary[['cluster_id', 'congestion_impact_score', 'total_incident_count']],
    on='cluster_id'
)

print(f"Slot-level rows: {len(slot)}")
avg_rows = slot.groupby('cluster_id').size().mean()
print(f"Avg rows per cluster: {avg_rows:.1f}")

Slot-level rows: 5958
Avg rows per cluster: 19.9


In [4]:
cluster_median = slot.groupby('cluster_id')['violation_count'].transform('median')
slot['label'] = (slot['violation_count'] > cluster_median).astype(int)  # 1=HIGH, 0=LOW

print(slot['label'].value_counts())
high_share = slot['label'].mean()
print(f"Class balance: {high_share:.1%} HIGH")

label
0    3229
1    2729
Name: count, dtype: int64
Class balance: 45.8% HIGH


In [5]:
day_map = {d: i for i, d in enumerate(
    ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
)}
bucket_map = {b: i for i, b in enumerate(
    ['early_morning','late_morning','afternoon','evening','night']
)}
slot['day_encoded'] = slot['cdt_day_of_week'].map(day_map)
slot['bucket_encoded'] = slot['time_bucket'].map(bucket_map)

FEATURES = ['day_encoded', 'bucket_encoded', 'congestion_impact_score', 'total_incident_count']
X = slot[FEATURES]
y = slot['label']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(
    n_estimators=200, max_depth=6, class_weight='balanced', random_state=42
)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("F1 (macro):", round(f1_score(y_test, pred, average='macro'), 3))
print()
print(classification_report(y_test, pred, target_names=['LOW', 'HIGH']))

cv_scores = cross_val_score(clf, X, y, cv=5)
print(f"5-fold CV accuracy: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

print()
print("Feature importances:")
for f, imp in sorted(zip(FEATURES, clf.feature_importances_), key=lambda x: -x[1]):
    print(f"  {f}: {imp:.3f}")

Accuracy: 0.654
F1 (macro): 0.652

              precision    recall  f1-score   support

         LOW       0.69      0.67      0.68       646
        HIGH       0.62      0.64      0.63       546

    accuracy                           0.65      1192
   macro avg       0.65      0.65      0.65      1192
weighted avg       0.65      0.65      0.65      1192

5-fold CV accuracy: 0.653 +/- 0.055

Feature importances:
  bucket_encoded: 0.598
  congestion_impact_score: 0.173
  total_incident_count: 0.172
  day_encoded: 0.058


## Tier 3: Anomaly Detection (z-score)

For each cluster, compute the historical mean/std of its own daily
violation volume, then flag any cluster-day whose actual count is an
unusual spike relative to that cluster's own baseline (z-score > 2.5).
Fully explainable -- no black box, easy to defend: "this day was N
standard deviations above what's normal for this specific hotspot."
This is the signal most directly relevant to the PS's framing around
event-driven/unplanned congestion.

In [7]:
df_clusters['cdt_date'] = pd.to_datetime(df_clusters['created_datetime']).dt.date

daily = (
    df_clusters.groupby(['cluster_id', 'cdt_date'])
    .agg(daily_count=('id', 'nunique'))
    .reset_index()
)

cluster_stats = daily.groupby('cluster_id')['daily_count'].agg(['mean', 'std', 'count']).reset_index()
cluster_stats.columns = ['cluster_id', 'baseline_mean', 'baseline_std', 'n_days_observed']

n_thin = (cluster_stats['n_days_observed'] < 2).sum()
print(f"{n_thin} cluster(s) have <2 days of data -- baseline std is undefined for these, "
      f"handled below by treating them as not-anomalous rather than producing NaN z-scores")

daily = daily.merge(cluster_stats, on='cluster_id')

daily['z_score'] = np.where(
    (daily['baseline_std'] > 0) & daily['baseline_std'].notna(),
    (daily['daily_count'] - daily['baseline_mean']) / daily['baseline_std'],
    0.0
)

ANOMALY_THRESHOLD = 2.5
daily['is_anomalous'] = daily['z_score'] > ANOMALY_THRESHOLD

n_anom = daily['is_anomalous'].sum()
print(f"Anomalous cluster-days flagged: {n_anom} / {len(daily)} ({n_anom/len(daily)*100:.2f}%)")
print()
print("Top flagged anomalies:")
print(daily[daily['is_anomalous']].sort_values('z_score', ascending=False)
      [['cluster_id','cdt_date','daily_count','baseline_mean','z_score']].head(10))

0 cluster(s) have <2 days of data -- baseline std is undefined for these, handled below by treating them as not-anomalous rather than producing NaN z-scores
Anomalous cluster-days flagged: 531 / 15888 (3.34%)

Top flagged anomalies:
       cluster_id    cdt_date  daily_count  baseline_mean   z_score
5932           62  2024-01-24          134       7.411111  8.650308
3858           38  2023-11-27           78       8.619048  8.301279
4969           51  2024-02-04          143      12.969925  7.943314
5815           61  2024-01-30          400      60.079470  7.365124
6482           70  2024-02-13          121      11.588785  6.983051
3799           36  2024-03-08           78       7.552239  6.943177
9849          114  2024-02-23          110       7.101449  6.753419
7276           77  2024-04-02           90      10.457627  6.568132
106             1  2024-01-12           60       6.594937  6.415498
12533         175  2023-12-13           41       3.400000  6.310315


In [8]:
def predict_priority_risk(cluster_id, day_of_week, hour, clf=clf, cluster_summary=cluster_summary):
    row = cluster_summary[cluster_summary['cluster_id'] == cluster_id]
    if row.empty:
        return {"cluster_id": cluster_id, "error": "cluster not found"}

    bucket = coarse_bucket(hour)
    features = pd.DataFrame([{
        'day_encoded': day_map.get(day_of_week, -1),
        'bucket_encoded': bucket_map.get(bucket, -1),
        'congestion_impact_score': row['congestion_impact_score'].values[0],
        'total_incident_count': row['total_incident_count'].values[0],
    }])[FEATURES]

    pred_label = clf.predict(features)[0]
    pred_proba = clf.predict_proba(features)[0]

    return {
        "cluster_id": int(cluster_id), "day_of_week": day_of_week, "hour": int(hour),
        "time_bucket": bucket, "predicted_risk": "HIGH" if pred_label == 1 else "LOW",
        "confidence": round(float(max(pred_proba)), 3)
    }


def check_anomaly(cluster_id, date, cluster_stats=cluster_stats):
    stats_row = cluster_stats[cluster_stats['cluster_id'] == cluster_id]
    if stats_row.empty:
        return {"cluster_id": cluster_id, "error": "cluster not found"}

    match = daily[(daily['cluster_id'] == cluster_id) & (daily['cdt_date'] == date)]
    if match.empty:
        return {
            "cluster_id": int(cluster_id), "date": str(date),
            "error": "no observed count for this cluster/date combination"
        }

    r = match.iloc[0]
    return {
        "cluster_id": int(cluster_id), "date": str(date),
        "actual_count": int(r['daily_count']),
        "baseline_mean": round(float(r['baseline_mean']), 2),
        "z_score": round(float(r['z_score']), 2),
        "is_anomalous": bool(r['is_anomalous'])
    }


# Sanity checks
top_cluster_id = int(cluster_summary.sort_values('congestion_impact_score', ascending=False).iloc[0]['cluster_id'])
print(predict_priority_risk(top_cluster_id, "Monday", 10))
print(predict_priority_risk(top_cluster_id, "Monday", 3))

anomaly_example = daily[daily['is_anomalous']].iloc[0]
print(check_anomaly(int(anomaly_example['cluster_id']), anomaly_example['cdt_date']))

{'cluster_id': 2, 'day_of_week': 'Monday', 'hour': 10, 'time_bucket': 'late_morning', 'predicted_risk': 'HIGH', 'confidence': 0.861}
{'cluster_id': 2, 'day_of_week': 'Monday', 'hour': 3, 'time_bucket': 'early_morning', 'predicted_risk': 'HIGH', 'confidence': 0.661}
{'cluster_id': 0, 'date': '2023-12-22', 'actual_count': 47, 'baseline_mean': 8.05, 'z_score': 3.31, 'is_anomalous': True}


## Persist the trained model + anomaly baselines

Saved as a single pickle (per request) containing the classifier, its
encoders, and each cluster's anomaly baseline stats -- one artifact a
backend can load once and use for both live Tier 2 predictions and live
Tier 3 anomaly checks.

In [ ]:
MODEL_PATH = "../../data/model/risk_classifier_bundle.pkl"

model_bundle = {
    "classifier": clf,
    "day_map": day_map,
    "bucket_map": bucket_map,
    "features": FEATURES,
    "anomaly_baselines": cluster_stats.set_index('cluster_id')[
        ['baseline_mean', 'baseline_std', 'n_days_observed']
    ].to_dict(orient='index'),
    "anomaly_threshold": ANOMALY_THRESHOLD,
    "trained_on_n_clusters": int(cluster_summary['cluster_id'].nunique()),
}

with open(MODEL_PATH, "wb") as f:
    pickle.dump(model_bundle, f)

print(f"Pickled classifier + anomaly baselines to {MODEL_PATH}")

Pickled classifier + anomaly baselines to ../../data/processed/risk_classifier_bundle.pkl


## Export: merge into `hotspots_details.json`

Adds two new keys per hotspot -- `priority_prediction` (Tier 2, full
precomputed day/bucket grid) and `anomaly_history` (Tier 3, flagged
anomalous dates only, not every day, to keep the file size reasonable).
Existing fields (including Tier 1's `risk_by_day_hour` if already run)
are untouched.

In [10]:
import json

DETAILS_PATH = "../../data/processed/hotspots_details.json"

with open(DETAILS_PATH, "r") as f:
    hotspots_details = json.load(f)

rows = []
for cid in cluster_summary['cluster_id'].unique():
    crow = cluster_summary[cluster_summary['cluster_id'] == cid].iloc[0]
    for day in day_map.keys():
        for bucket_name in bucket_map.keys():
            rows.append({
                'cluster_id': int(cid), 'day_of_week': day, 'time_bucket': bucket_name,
                'day_encoded': day_map[day], 'bucket_encoded': bucket_map[bucket_name],
                'congestion_impact_score': crow['congestion_impact_score'],
                'total_incident_count': crow['total_incident_count'],
            })

batch_df = pd.DataFrame(rows)
batch_preds = clf.predict(batch_df[FEATURES])
batch_probas = clf.predict_proba(batch_df[FEATURES])
batch_df['predicted_risk'] = np.where(batch_preds == 1, "HIGH", "LOW")
batch_df['confidence'] = batch_probas.max(axis=1).round(3)

print(f"Tier 2: batch-predicted {len(batch_df)} (cluster, day, bucket) combinations")

anomalies_only = daily[daily['is_anomalous']].copy()
anomalies_only['cdt_date'] = anomalies_only['cdt_date'].astype(str)
print(f"Tier 3: {len(anomalies_only)} anomalous cluster-days to export")

n_merged = 0
n_no_match = 0

for cid in cluster_summary['cluster_id'].unique():
    cid_str = str(int(cid))

    cid_preds = batch_df[batch_df['cluster_id'] == cid]
    predictions = {}
    for day, day_group in cid_preds.groupby('day_of_week'):
        predictions[day] = {
            row['time_bucket']: {
                "predicted_risk": row['predicted_risk'],
                "confidence": float(row['confidence'])
            }
            for _, row in day_group.iterrows()
        }

    cid_anomalies = anomalies_only[anomalies_only['cluster_id'] == cid]
    anomaly_list = [
        {
            "date": row['cdt_date'],
            "actual_count": int(row['daily_count']),
            "baseline_mean": round(float(row['baseline_mean']), 2),
            "z_score": round(float(row['z_score']), 2),
        }
        for _, row in cid_anomalies.iterrows()
    ]

    if cid_str in hotspots_details:
        hotspots_details[cid_str]["priority_prediction"] = predictions
        hotspots_details[cid_str]["anomaly_history"] = anomaly_list
        n_merged += 1
    else:
        n_no_match += 1

print(f"Merged priority_prediction + anomaly_history into {n_merged} hotspot entries")
if n_no_match:
    print(f"WARNING: {n_no_match} clusters had no matching entry in hotspots_details.json")

with open(DETAILS_PATH, "w") as f:
    json.dump(hotspots_details, f, indent=2)

print(f"Saved updated {DETAILS_PATH}")

Tier 2: batch-predicted 10500 (cluster, day, bucket) combinations
Tier 3: 531 anomalous cluster-days to export
Merged priority_prediction + anomaly_history into 300 hotspot entries
Saved updated ../../data/processed/hotspots_details.json
